# FirstAI / Orbit — persistent Colab data preparation and T4 training
Save collected data, run logs and model checkpoints in Google Drive so they survive Colab runtime resets. CPU is for data preparation; the existing PyTorch training cells require CUDA GPU.


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
drive.mount('/content/drive')  # Approve Google's access prompt
PERSIST_ROOT = Path('/content/drive/MyDrive/FirstAI_Orbit')
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
print('Persistent workspace:', PERSIST_ROOT, flush=True)
print('Data, logs and checkpoints will be saved in Google Drive.', flush=True)


In [ ]:
import os, subprocess, sys, time
from pathlib import Path
REPO=Path('/content/FirstAI')
if REPO.exists():
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
else:
    subprocess.run(['git','clone','https://github.com/llakshyaa-rgb/FirstAI.git',str(REPO)],check=True)
os.chdir(REPO); sys.path.insert(0,str(REPO))
# Persist large artifacts outside Colab's temporary runtime disk.
import shutil
for folder in ['data', 'checkpoints', 'run_logs']:
    local = REPO / folder
    persistent = PERSIST_ROOT / folder
    persistent.mkdir(parents=True, exist_ok=True)
    if local.exists() and not local.is_symlink():
        shutil.copytree(local, persistent, dirs_exist_ok=True)
        shutil.rmtree(local)
    if local.is_symlink():
        local.unlink()
    local.symlink_to(persistent, target_is_directory=True)
print('Persistent data, logs and checkpoints linked to Drive.', flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q','datasets>=3.0'],check=True)
import torch
print('PyTorch:',torch.__version__,'CUDA build:',torch.version.cuda,flush=True)
if torch.cuda.is_available():
    print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/1024**3,2),flush=True)
else:
    print('CPU MODE: data collection/preparation is ready. GPU is only required in the training cells.',flush=True)


## 1. Prepare base text, task data and multi-turn conversations
Downloads up to 30M WikiText characters, capability tasks, sourced public-data examples and up to 20,000 assistant turns from UltraChat 200k (training split only). Conversation rows preserve preceding turns so SFT teaches dialogue continuation, not just isolated one-shot instructions.


In [ ]:
import time, json
from pathlib import Path
print('DATA PREPARATION MODE: CPU is expected; no CUDA check is performed.',flush=True)
print('Live logs include stage names, per-source counts/speeds when provided, 30-second heartbeat messages during quiet downloads, and elapsed time. Keep this cell running until DATA PREPARATION VERIFIED appears.',flush=True)
started=time.time()
cmd=[sys.executable,'-u','build_all_data.py','--country','IND','--city','Patna','--max-per-dataset','10000','--max-base-chars','30000000','--per-source-cap','20000','--conversation-limit','20000']
log_path = REPO / 'run_logs' / f"data_build_{time.strftime('%Y%m%d_%H%M%S')}.log"
log_path.parent.mkdir(parents=True, exist_ok=True)
print('Full live log:', log_path, flush=True)
with log_path.open('a', encoding='utf-8') as log:
    process = subprocess.Popen(cmd, cwd=REPO, env={**os.environ,'PYTHONUNBUFFERED':'1'},
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, cmd)
print('Build progress manifest:', REPO / 'run_logs' / 'build_progress.json', flush=True)
print(f'\nData preparation elapsed: {(time.time()-started)/60:.1f} minutes',flush=True)
for f in ['data/raw/wikitext_train.txt','data/input.txt','data/capabilities/manifest.json','data/instructions_conversations.manifest.json','data/instructions_balanced.manifest.json']:
    p=Path(f)
    print(f'{f}: {p.stat().st_size:,} bytes' if p.exists() else f'{f}: MISSING',flush=True)
if not Path('data/input.txt').exists() or Path('data/input.txt').stat().st_size==0:
    raise RuntimeError('Base corpus was not created. Do not start training.')
if not Path('data/instructions_balanced.jsonl').exists() or Path('data/instructions_balanced.jsonl').stat().st_size==0:
    raise RuntimeError('Balanced instruction corpus was not created. Do not start fine-tuning.')
print('\nDATA PREPARATION VERIFIED. Switch to T4 GPU before running training.',flush=True)


## 2. Train a substantially larger FirstGPT from scratch
This profile uses 8 layers, width 512, 8 attention heads, a 4,096-token BPE vocabulary and 1,024-token context (approximately 27M parameters, depending on the learned vocabulary size). It is still a small experimental model, not a frontier model. T4 GPU is required; 12,000 steps may take hours and Colab session limits are not guaranteed.


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU REQUIRED FOR TRAINING: stop here, select Runtime > Change runtime type > T4 GPU, reconnect, rerun setup, then rerun this cell. Do not train this model on CPU.')
print('Training GPU:',torch.cuda.get_device_name(0),flush=True)
base_args=['--data','data/input.txt','--out','checkpoints/firstgpt-convo.pt','--steps',os.environ.get('FIRSTAI_BASE_STEPS','12000'),'--eval-interval','200','--checkpoint-interval','500','--eval-batches','8','--batch-size','1','--grad-accum-steps','16','--block-size','1024','--layers','8','--heads','8','--width','512','--vocab-size','4096','--tokenizer-train-chars','100000','--architecture-version','2','--amp']
if Path('checkpoints/firstgpt-convo.pt').exists() and os.environ.get('FIRSTAI_RESTART_BASE','0')!='1': base_args.append('--resume')
subprocess.run([sys.executable,'-u','train.py',*base_args],check=True,cwd=REPO)


## 3. Supervised conversational instruction tuning
This stage mixes multi-turn assistant conversations with sourced mathematics, science, reading, coding and Orbit behaviour examples. It trains only on assistant response tokens; user prompts are context, not target text. Resume is allowed only when the dataset hash matches.


In [ ]:
import torch
from pathlib import Path
if not torch.cuda.is_available():
    raise RuntimeError('GPU REQUIRED FOR FINE-TUNING: select T4 GPU and reconnect before running this cell.')
BASE_CHECKPOINT = Path('checkpoints/firstgpt-convo.best.pt')
SFT_DATA = Path('data/instructions_rebuilt_v3.jsonl')
SFT_OUT = Path('checkpoints/orbit-convo-instruct-v3.pt')
if not BASE_CHECKPOINT.is_file():
    raise FileNotFoundError(f'Missing protected base checkpoint: {BASE_CHECKPOINT}')
if not SFT_DATA.is_file():
    raise FileNotFoundError(f'Missing audited V3 dataset: {SFT_DATA}. Build and audit V3 first; do not fall back to the old balanced dataset.')
if SFT_OUT.exists():
    raise FileExistsError(f'{SFT_OUT} already exists. Use a deliberate resume only after confirming its dataset hash, or choose a new output filename.')
print('Fine-tuning GPU:', torch.cuda.get_device_name(0), flush=True)
print('Base checkpoint:', BASE_CHECKPOINT, BASE_CHECKPOINT.stat().st_size, 'bytes', flush=True)
print('SFT dataset:', SFT_DATA, SFT_DATA.stat().st_size, 'bytes', flush=True)
sft_args = ['--checkpoint', str(BASE_CHECKPOINT), '--data', str(SFT_DATA), '--out', str(SFT_OUT), '--steps', os.environ.get('FIRSTAI_SFT_STEPS','6000'), '--batch-size', '1', '--eval-interval', '100', '--lr', '0.00002']
subprocess.run([sys.executable, '-u', 'finetune_instructions.py', *sft_args], check=True, cwd=REPO)
best = SFT_OUT.with_name(SFT_OUT.stem + '.best' + SFT_OUT.suffix)
if not best.is_file(): raise FileNotFoundError('SFT did not produce the expected V3 best checkpoint.')
print('Best V3 SFT checkpoint:', best.stat().st_size, 'bytes', flush=True)


## 4. Inspect real replies before deploying
This is a qualitative smoke test, not an automatic proof of intelligence. Read the answers: if they loop, ignore context or invent facts, do not deploy the checkpoint as ready.


In [ ]:
from orbit.runtime import OrbitRuntime
runtime=OrbitRuntime('checkpoints/orbit-convo-instruct.best.pt',max_new_tokens=100)
history=[]
for index,prompt in enumerate(['My project is called Nebula.','What did I call my project?','And what is it called?']):
 answer=runtime.reply(prompt,history=history)
 print('\nOWNER:',prompt,'\nORBIT:',answer)
 if not answer.strip(): raise RuntimeError('Empty answer. Do not deploy this checkpoint.')
 if index == 1 and 'nebula' not in answer.lower(): raise RuntimeError('Multi-turn recall failed. Do not deploy this checkpoint yet.')
 history.extend([{'role':'user','content':prompt},{'role':'assistant','content':answer}])
print('\nGENERAL TEST:',runtime.reply('Explain why the sky looks blue in simple terms.'))


## 5. Download the trained checkpoints
Download both files before disconnecting; Colab's temporary disk is erased when the runtime ends. Use the SFT best checkpoint for Orbit.


In [ ]:
from google.colab import files
files.download('checkpoints/firstgpt-convo.best.pt')
files.download('checkpoints/orbit-convo-instruct.best.pt')
